# Metrics, Losses & Sampling from Scratch

Metrics and losses are tested in every ML coding loop — precision/recall, AUC, focal loss, and various sampling schemes appear constantly. This note implements them all from first principles and maps problem types to the right metric choices.

## What Interviewers Test
- Implementing precision, recall, F1, and ROC-AUC without sklearn
- The two AUC computation methods: rank-statistic vs trapezoid
- Numerically stable log-loss implementation
- Focal loss and when to use it over cross-entropy
- Reservoir sampling for streaming scenarios
- Negative sampling for recommenders / embedding training

## Metric Selection Table

| Problem Type | Primary Metric | Secondary | Avoid |
|---|---|---|---|
| Balanced classification | Accuracy, F1 | Confusion matrix | — |
| Imbalanced classification | PR-AUC, F1 | ROC-AUC | Accuracy (misleading) |
| Ranking / retrieval | NDCG, MAP | MRR, Recall@K | Accuracy |
| Calibration | ECE, Brier score | Log-loss | ROC-AUC |
| Binary with high-FP cost | Precision@K | PR curve | F1 |
| Binary with high-FN cost | Recall | F-beta (β>1) | Precision |


In [ ]:
import numpy as np
from sklearn import metrics as skm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# --- Synthetic imbalanced data ---
n_pos, n_neg = 100, 900
y_true  = np.array([1]*n_pos + [0]*n_neg)
# Imperfect scores
scores  = np.concatenate([
    np.random.beta(5, 2, n_pos),     # positives: skewed high
    np.random.beta(2, 5, n_neg)      # negatives: skewed low
])
y_pred = (scores > 0.5).astype(int)


## Precision, Recall, F1 from Confusion Matrix

$$\text{Precision} = \frac{TP}{TP+FP}, \quad \text{Recall} = \frac{TP}{TP+FN}, \quad F_1 = \frac{2 \cdot P \cdot R}{P + R}$$


In [ ]:
def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    return TP, TN, FP, FN

def precision_recall_f1(y_true, y_pred, beta=1.0):
    TP, TN, FP, FN = confusion_matrix(y_true, y_pred)
    precision = TP / (TP + FP + 1e-12)
    recall    = TP / (TP + FN + 1e-12)
    f_beta    = (1 + beta**2) * precision * recall / (beta**2 * precision + recall + 1e-12)
    return precision, recall, f_beta

p, r, f1 = precision_recall_f1(y_true, y_pred)
sk_p  = skm.precision_score(y_true, y_pred)
sk_r  = skm.recall_score(y_true, y_pred)
sk_f1 = skm.f1_score(y_true, y_pred)

print(f"Precision  — scratch: {p:.4f}, sklearn: {sk_p:.4f}")
print(f"Recall     — scratch: {r:.4f}, sklearn: {sk_r:.4f}")
print(f"F1         — scratch: {f1:.4f}, sklearn: {sk_f1:.4f}")


## ROC-AUC: Two Methods

**Method 1 (rank statistic):** $\text{AUC} = \frac{\text{(sum of ranks of positives)} - n_+(n_++1)/2}{n_+ \cdot n_-}$  → O(n log n)

**Method 2 (trapezoid):** Compute TPR/FPR at all thresholds, apply trapezoid rule → O(n log n)

> 💡 **Interview Tip:** The rank-statistic interpretation is powerful: AUC = P(score of random positive > score of random negative). This makes AUC meaningful even when the threshold isn't calibrated.


In [ ]:
def roc_auc_rank(y_true, scores):
    """Wilcoxon-Mann-Whitney rank statistic — O(n log n)."""
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    # Rank all scores (1-indexed). Average ties.
    order = np.argsort(scores)
    ranks = np.empty(len(scores))
    ranks[order] = np.arange(1, len(scores)+1)
    # Handle ties: average rank within tied groups
    sorted_scores = scores[order]
    unique, counts = np.unique(sorted_scores, return_counts=True)
    start = 0
    for cnt in counts:
        ranks[order[start:start+cnt]] = ranks[order[start:start+cnt]].mean()
        start += cnt
    rank_sum_pos = ranks[y_true == 1].sum()
    auc = (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc

def roc_auc_trapezoid(y_true, scores):
    """Trapezoid rule on TPR/FPR curve."""
    thresholds = np.sort(np.unique(scores))[::-1]
    n_pos = y_true.sum(); n_neg = len(y_true) - n_pos
    tprs = [0.0]; fprs = [0.0]
    for t in thresholds:
        pred = (scores >= t).astype(int)
        TP = np.sum((pred == 1) & (y_true == 1))
        FP = np.sum((pred == 1) & (y_true == 0))
        tprs.append(TP / (n_pos + 1e-12))
        fprs.append(FP / (n_neg + 1e-12))
    tprs.append(1.0); fprs.append(1.0)
    tprs, fprs = np.array(tprs), np.array(fprs)
    # NumPy 2.0 removed np.trapz and renamed it np.trapezoid.
    # This keeps the notebook running on both 1.x and 2.x.
    trapezoid = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return trapezoid(tprs, fprs)

auc_rank = roc_auc_rank(y_true, scores)
auc_trap = roc_auc_trapezoid(y_true, scores)
auc_sk   = skm.roc_auc_score(y_true, scores)

print(f"AUC rank statistic : {auc_rank:.4f}")
print(f"AUC trapezoid      : {auc_trap:.4f}")
print(f"AUC sklearn        : {auc_sk:.4f}")


## Losses: Log-Loss, Weighted CE, Focal Loss

**Focal loss** (Lin et al., 2017): $FL(p_t) = -(1-p_t)^\gamma \log(p_t)$

It down-weights easy examples (high $p_t$) and focuses training on hard ones. Used in object detection (RetinaNet) and class-imbalanced settings.


In [ ]:
def log_loss_scratch(y_true, y_prob, eps=1e-15):
    y_prob = np.clip(y_prob, eps, 1-eps)
    return -np.mean(y_true * np.log(y_prob) + (1-y_true) * np.log(1-y_prob))

def weighted_cross_entropy(y_true, y_prob, class_weights, eps=1e-15):
    """class_weights: (n_classes,) or per-sample weights."""
    y_prob = np.clip(y_prob, eps, 1-eps)
    w = class_weights[y_true.astype(int)]
    return -np.mean(w * (y_true * np.log(y_prob) + (1-y_true) * np.log(1-y_prob)))

def focal_loss(y_true, y_prob, gamma=2.0, eps=1e-15):
    """Focal loss for binary classification."""
    y_prob = np.clip(y_prob, eps, 1-eps)
    p_t  = np.where(y_true == 1, y_prob, 1 - y_prob)
    fl   = -(1 - p_t)**gamma * np.log(p_t)
    return fl.mean()

y_prob = scores  # use our synthetic scores as probabilities

ll_scratch = log_loss_scratch(y_true, y_prob)
ll_sk      = skm.log_loss(y_true, y_prob)
print(f"Log-loss  — scratch: {ll_scratch:.4f}, sklearn: {ll_sk:.4f}")

# Class weights to handle imbalance (weight positives higher)
cw = np.array([1.0, 9.0])  # [neg_weight, pos_weight]
wce = weighted_cross_entropy(y_true, y_prob, cw)
print(f"Weighted CE (9x pos weight): {wce:.4f}")

fl_g0 = focal_loss(y_true, y_prob, gamma=0)   # same as standard BCE
fl_g2 = focal_loss(y_true, y_prob, gamma=2)
fl_g5 = focal_loss(y_true, y_prob, gamma=5)
print(f"Focal loss γ=0 (=BCE): {fl_g0:.4f}")
print(f"Focal loss γ=2:         {fl_g2:.4f}")
print(f"Focal loss γ=5:         {fl_g5:.4f}")
print("Higher γ → more focus on hard examples → lower average loss (discounts easy ones)")


## Sampling Algorithms

### Reservoir Sampling
Maintain a sample of size $k$ from an online stream of unknown length $N$: every new element $i$ is included with probability $k/i$.

> 💡 **Interview Tip:** Reservoir sampling is asked when the interviewer mentions "streaming data" or "you can't fit the dataset in memory." It guarantees a uniform random sample without knowing N in advance.


In [ ]:
import random

def reservoir_sample(stream, k):
    """O(n) time, O(k) space. stream: any iterable."""
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = item
    return reservoir

def weighted_sample(items, weights, k, replace=False):
    """Sample k items with given weights."""
    weights = np.array(weights, dtype=float)
    probs = weights / weights.sum()
    indices = np.random.choice(len(items), size=k, replace=replace, p=probs)
    return [items[i] for i in indices]

def stratified_split(X, y, test_size=0.2):
    """Maintain class proportions in train/test split."""
    classes = np.unique(y)
    train_idx, test_idx = [], []
    for c in classes:
        idx = np.where(y == c)[0]
        np.random.shuffle(idx)
        n_test = max(1, int(len(idx) * test_size))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    return np.array(train_idx), np.array(test_idx)

def negative_sample(pos_pairs, n_items, n_neg_per_pos=5, pop_weights=None):
    """
    For each (user, pos_item) pair, sample n_neg_per_pos negative items.
    pop_weights: item popularity for popularity-biased negative sampling.
    """
    if pop_weights is None:
        pop_weights = np.ones(n_items)
    pop_weights = pop_weights / pop_weights.sum()
    
    result = []
    for user, pos_item in pos_pairs:
        neg_items = []
        attempts = 0
        while len(neg_items) < n_neg_per_pos and attempts < n_neg_per_pos * 10:
            cand = np.random.choice(n_items, p=pop_weights)
            if cand != pos_item:
                neg_items.append(cand)
            attempts += 1
        result.append((user, pos_item, neg_items))
    return result

# --- Test reservoir sampling ---
stream = list(range(1000))
sample = reservoir_sample(stream, k=100)
print(f"Reservoir sample size: {len(sample)}")
print(f"Min: {min(sample)}, Max: {max(sample)} (should cover full range uniformly)")

# --- Verify stratification ---
y_strat = np.array([0]*900 + [1]*100)
tr_idx, te_idx = stratified_split(np.arange(1000), y_strat, test_size=0.2)
print(f"\nStratified split — train pos rate: {y_strat[tr_idx].mean():.3f}, test pos rate: {y_strat[te_idx].mean():.3f}")
print(f"vs. naive random: {y_strat.mean():.3f}")


## Common Interview Questions

**Q: Why is accuracy a bad metric for fraud detection (0.1% fraud rate)?**
A model predicting "never fraud" achieves 99.9% accuracy while being completely useless. Use precision/recall/F1 on the fraud class, or PR-AUC, which explicitly focuses on the minority class performance.

**Q: What does AUC-ROC actually measure?**
The probability that a randomly chosen positive example receives a higher score than a randomly chosen negative example. An AUC of 0.5 is random; 1.0 is perfect. Importantly, it's threshold-independent and scale-invariant — only the ranking matters, not the calibration.

**Q: When would you prefer PR-AUC over ROC-AUC?**
When the negative class vastly outnumbers the positive (extreme imbalance). ROC-AUC can look optimistic because the large TN count keeps FPR low even with many false positives. PR-AUC focuses on how many retrieved positives are actually positive, which is the relevant question in imbalanced settings.

**Q: What is focal loss and when do you use it?**
Focal loss adds a $(1-p_t)^\gamma$ modulating factor to cross-entropy. Easy examples (high $p_t$) get down-weighted, so the loss focuses on hard or misclassified examples. Use it for extreme class imbalance (e.g., object detection background vs objects) or when the model converges but struggles on hard cases.

**Q: What is reservoir sampling and why is it useful?**
It produces a uniform random sample of size $k$ from a stream of unknown length in O(n) time and O(k) space. Useful when data arrives as a stream or the full dataset doesn't fit in memory. Each element gets probability $k/i$ of being included when it's the $i$th element.

## Key Takeaways
- Precision = TP/(TP+FP), Recall = TP/(TP+FN); F1 is their harmonic mean
- AUC = P(positive scores higher than negative) — threshold-independent ranking quality
- For extreme imbalance: prefer PR-AUC over ROC-AUC and consider weighted losses
- Focal loss down-weights easy examples by $(1-p_t)^\gamma$; higher $\gamma$ = more focus on hard cases
- Reservoir sampling: O(n) stream sampling in O(k) space, provably uniform
- Stratified splits preserve class ratios — critical for imbalanced datasets
- Always clip probabilities before log: `np.clip(p, 1e-15, 1-1e-15)`